# Managing Credits

In [1]:
from arcgis.gis import GIS

In [2]:
gis = GIS(profile='your_online_profile')

## Credit Settings

- defining default is your first line of defense for protecting your organizations credits
- By defaults, everyone can consume all credits

In [4]:
credit_manager = gis.admin.credits

### Check the Organization's Available Credits

In [5]:
credit_manager.credits

3098223.5

### Enable Credit Management

In [6]:
if credit_manager.is_enabled == False:
    credit_manager.enable()
print(f"Is credit management enabled? {credit_manager.is_enabled}")

Is credit management enabled? True


### Set Default Credits for Users

In [7]:
if credit_manager.default_limit == -1:
    credit_manager.default_limit = 600
print(f"The default credit limit is set to {credit_manager.default_limit}")

The default credit limit is not set to 601


### Allocating and Deallocating Credits to Users

- When you want to set a given amount of credits to a user, use the `allocate` method
- When you want to give unlimited credits for a user, use the `deallocate` method

In [8]:
import uuid
username = f"RUser{uuid.uuid4().hex[:4]}"
password = f"!{uuid.uuid4().hex[:8]}A"
um = gis.users
new_user = um.create(username=username, password=password, 
                     firstname="Dan", lastname="Human", 
                     email='testsadf@esri.com', 
                     role="org_publisher")
new_user

<User username:RUser7404>

In [9]:
print(f"This new user has {new_user.assignedCredits} credits available")

This new user has 601.0 credits available


In [10]:
credit_manager.deallocate(new_user.username)

True

In [11]:
new_user = gis.users.get(new_user.username)
print(f"This new user has {new_user.assignedCredits} credits available")

This new user has -1.0 credits available


In [12]:
credit_manager.allocate(new_user.username, 50)
new_user = gis.users.get(new_user.username)
print(f"This new user has {new_user.assignedCredits} credits available")

This new user has 50.0 credits available


In [13]:
new_user.delete()

True

## Reporting on Credits

- Administrators can create reports on multiple things, but one is specific to credit usage of the organization.
- The start time depends on when the `duration` of the report

In [14]:
from arcgis.gis import User
import datetime as _dt
admin_user:User = gis.users.me

In [15]:
item_report = admin_user.report("credits", start_time=_dt.datetime(2025, 6, 1), duration='monthly')
item_report

<Item title:"OrganizationCreditsMonthly_2025-06" type:Administrative Report owner:nparavicini_geosaurus>

In [16]:
import pandas as pd

In [17]:
data = item_report.get_data()
df = pd.read_csv(data, skiprows=3)
df.head(10)

,Credits Consumed by,App Title,ArcGIS Notebook - Interactive,ArcGIS Notebook - Scheduled,Closest Facility Routes,Demographic Maps,Feature Reports,Geocoding,GeoEnrichment,Location-Allocation,...,Imagery Analysis,Feature Storage,File Storage,Imagery Storage,Storage in an ArcGIS Notebooks,Scene Storage,Tile Storage,Vector Tiles Storage,ModelBuilder - Interactive,Snap to Road
0,ArcGISPyAPIBot,NaN,48.35,4.35,0.0,0.00,0.0,0.00,0.00,0.0,...,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0
1,ArcGISPyAPIBot,python api,0.00,0.00,0.0,0.00,0.0,138.04,0.00,0.0,...,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0
2,NGiner_geosaurus,NaN,0.50,0.00,0.0,0.00,0.0,6.00,0.00,0.0,...,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0
3,NGiner_geosaurus,ArcGIS Pro,0.00,0.00,0.0,7.05,0.0,0.00,151.85,0.0,...,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0
4,NGiner_geosaurus,python api,0.00,0.00,0.0,0.00,0.0,17.40,0.00,0.0,...,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0
5,Python API Test,NaN,0.00,0.00,0.0,0.00,0.0,0.00,0.00,0.0,...,0.0,11241.63,284.36,299.03,0.02,10.84,2.47,0.21,0.0,0.0
6,andrew57,NaN,0.00,13.93,2.0,0.00,0.0,0.00,0.00,0.0,...,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0
7,arcgis_python,NaN,0.00,0.00,0.0,0.00,0.0,54.36,0.00,0.0,...,2.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0
8,arcgis_python,python api,0.00,0.00,4.0,0.00,0.0,90.36,1992.12,0.0,...,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0
9,jyaist_geosaurus,NaN,2.25,0.00,0.0,0.00,0.0,0.00,0.00,0.0,...,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0


In [18]:
item_report.delete(permanent=True)

True